In [59]:
# Dependencies
import pandas as pd
import os
import re
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from collections import Counter, defaultdict
import math

# Base Directory
base_dir = os.path.abspath("/Users/amberteetsel/MSDS/NLP/nlp-author-identification/")

## Tokenization

The tokenizer wrapper (`BPETokenizer` class) is original code that provides a simplified interface for the pipeline. The underlying BPE model, pre-tokenizers, and trainer are Hugging Face's `tokenizers` library implementations, used as instructed in the assignment (not reimplemented).

In [60]:
class BPETokenizer:
    """Wraps HuggingFace's tokenizers library to provide a simple
    encode/decode/train interface for n-gram LM pipeline."""

    def __init__(self, vocab_size=5000, pre_tokenizer="whitespace"):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

        if pre_tokenizer == "whitespace":
            self.tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
        elif pre_tokenizer == "byte_level":
            self.tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
        else:
            raise ValueError(f"Unknown pre_tokenizer: {pre_tokenizer}")

        self.special_tokens = ["<unk>", "<pad>", "<s>", "</s>"]

    def train(self, filepaths):
        """Train BPE on one or more text files (paths as a list)."""
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.special_tokens,
        )
        self.tokenizer.train(filepaths, trainer)

    def encode(self, text):
        """Returns list of token ids."""
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        """Returns string from list of token ids."""
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

In [61]:
# Training
hobbit_train_path = os.path.join(base_dir, "texts", "hobbit_train.txt")
lostworld_train_path = os.path.join(base_dir, "texts", "lostworld_train.txt")

bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="whitespace")
bpe.train([hobbit_train_path, lostworld_train_path])

In [62]:
sample = "Bilbo Baggins was a hobbit who lived in a hole in the ground."
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[239, 723, 117, 55, 474, 287, 1229, 88, 55, 1159, 88, 87, 685, 12]
Bilbo Baggins was a hobbit who lived in a hole in the ground .
vocab size: 5000


In [63]:
sample = """
Well, at least
you are better than that herd of swine in Vienna, whose gregarious
grunt is, however, not more offensive than the isolated effort of the
British hog.
"""
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[674, 10, 94, 981, 132, 201, 913, 357, 126, 1050, 58, 99, 1040, 425, 88, 4578, 3280, 10, 2156, 225, 1846, 741, 255, 2643, 105, 10, 1015, 10, 161, 275, 3923, 357, 87, 4110, 3161, 99, 87, 2606, 3552, 159, 61, 12]
Well , at least you are better than that her d of sw ine in Vien na , whose gre gar ious gr unt is , however , not more offensive than the isolated effort of the Br itish ho g .
vocab size: 5000


The decode is inserting a space between every token, even subword pieces that should be merged (e.g. "her d" instead of "herd"). This is the default behavior of the `Whitespace()` pre-tokenizer, so we'll move to the `ByteLevel` implementation.

After one iteration of `ByteLevel`, we discovered that any character that happened to not appear in training (e.g. "\n") will fall back to `<unk>` and cause incorrect spacing (merging words that should be separate). To fix, we specify that the trainer should start out with all 256 byte tokens for alphabet up front, instead of only what it happens to see in training corpus.

In [64]:
class BPETokenizer:
    def __init__(self, vocab_size=5000, pre_tokenizer="byte_level"):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

        if pre_tokenizer == "byte_level":
            self.tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
            self.tokenizer.decoder = decoders.ByteLevel()
        elif pre_tokenizer == "whitespace":
            self.tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
            # no matching decoder so use default
        else:
            raise ValueError(f"Unknown pre_tokenizer: {pre_tokenizer}")

        self.special_tokens = ["<unk>", "<pad>", "<s>", "</s>"]

    def train(self, filepaths):
        """Train BPE on one or more text files (paths as a list)."""
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.special_tokens,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
        )
        self.tokenizer.train(filepaths, trainer)

    def encode(self, text):
        """Normalize input and returns list of token ids."""
        text = re.sub(r"\s+", " ", text).strip()
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        """Returns string from list of token ids."""
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

In [65]:
# Training
hobbit_train_path = os.path.join(base_dir, "texts", "hobbit_train.txt")
lostworld_train_path = os.path.join(base_dir, "texts", "lostworld_train.txt")

bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="byte_level")
bpe.train([hobbit_train_path, lostworld_train_path])

In [66]:
sample = "Bilbo Baggins was a hobbit who lived in a hole in the ground."
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[4654, 981, 309, 262, 735, 545, 1570, 296, 262, 1982, 296, 263, 1103, 17]
Bilbo Baggins was a hobbit who lived in a hole in the ground.
vocab size: 5000


In [67]:
sample = """
Well, at least
you are better than that herd of swine in Vienna, whose gregarious
grunt is, however, not more offensive than the isolated effort of the
British hog.
"""
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[1090, 15, 366, 1285, 332, 468, 1207, 604, 325, 1565, 71, 280, 662, 670, 296, 3364, 2146, 3726, 15, 2552, 753, 2707, 656, 537, 407, 87, 379, 15, 1311, 15, 360, 513, 616, 3905, 604, 263, 4650, 3304, 280, 263, 3217, 4342, 481, 74, 17]
Well, at least you are better than that herd of swine in Vienna, whose gregarious grunt is, however, not more offensive than the isolated effort of the British hog.
vocab size: 5000


In [68]:
# Save trained tokenizer
bpe.save(os.path.join(base_dir, "src", "bpe_tokenizer.json"))

## N-Gram Language Model

In [69]:
# Get sentences from input text, same regex used in split_train_holdout
def get_sentences(text):
    """Splits text into sentences using regex."""
    s = re.sub(r"\s+", " ", text).strip()
    return re.split(r"(?<=[.!?])\s+(?=[A-Z])", s)

# Encode sentences, add <s> and </s> tokens
def encode_sentences(sentences, tokenizer):
    """Tokenizes each sentence and wraps with <s> and </s>"""
    s_id = tokenizer.tokenizer.token_to_id("<s>")
    e_id = tokenizer.tokenizer.token_to_id("</s>")
    encoded_sentences = []
    for sent in sentences:
        ids = tokenizer.encode(sent)
        if ids:
            encoded_sentences.append([s_id] + ids + [e_id])
    return encoded_sentences

In [70]:
# Read training text files as strings
with open(hobbit_train_path, encoding='utf-8') as f:
    hobbit_train = f.read()

with open(lostworld_train_path, encoding='utf-8') as f:
    lostworld_train = f.read()

In [71]:
# Encode sentences from hobbit and lostworld
hobbit_sentences = get_sentences(hobbit_train)
hobbit_encoded = encode_sentences(hobbit_sentences, bpe)

lostworld_sentences = get_sentences(lostworld_train)
lostworld_encoded = encode_sentences(lostworld_sentences, bpe)

In [72]:
# N-Gram model
class NGramModel:
    "Unigram, bigram and trigram counts with add-k smoothing"

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.unigram_counts = Counter()
        self.bigram_counts = Counter()
        self.trigram_counts = Counter()
        self.bigram_context_counts = Counter()
        self.trigram_context_counts = Counter()

    def train(self, encoded_sentences):
        """encoded_sentences: list of token id lists wrapped with <s> and </s>"""
        for sent in encoded_sentences:
            for i, token in enumerate(sent):
                self.unigram_counts[token] += 1

                if i >= 1:
                    bigram = (sent[i-1], sent[i])
                    self.bigram_counts[bigram] += 1
                    self.bigram_context_counts[(sent[i-1]),] += 1

                if i >= 2:
                    trigram = (sent[i-2], sent[i-1], sent[i])
                    self.trigram_counts[trigram] += 1
                    self.trigram_context_counts[(sent[i-2], sent[i-1])] += 1

    def bigram_prob(self, w1, w2, k=1.0):
        """P(w2 | w1) with add-k smoothing"""
        count_bigram = self.bigram_counts[(w1, w2)]
        count_context = self.bigram_context_counts[(w1,)]
        return (count_bigram + k) / (count_context + k * self.vocab_size)

    def trigram_prob(self, w1, w2, w3, k=1.0):
        """P(w3 | w1, w2) with add-k smoothing"""
        count_trigram = self.trigram_counts[(w1, w2, w3)]
        count_context = self.trigram_context_counts[(w1,w2)]
        return (count_trigram + k) / (count_context + k * self.vocab_size)

    def neg_log_prob(self, sent, n=2, k=1.0):
        """Returns total -logP(sentence) using n_gram model (n=2 for bigram, n=3 for trigram)"""
        total = 0.0
        for i in range(1, len(sent)) if n==2 else range(2, len(sent)):
            if n==2:
                p = self.bigram_prob(sent[i-1], sent[i], k)
            elif n==3:
                p = self.trigram_prob(sent[i-2], sent[i-1], sent[i], k)
            else:
                raise ValueError("n must be 2 (bigram) or 3 (trigram)")
            total += -math.log(p)
        return total

    def perplexity(self, sentences, n=2, k=1.0):
        """Computes perplexity over list of encoded sentences (list of token id lists)"""
        total_neg_log_prob = 0.0
        total_tokens = 0

        for sent in sentences:
            total_neg_log_prob += self.neg_log_prob(sent, n=n, k=k)
            # count predicted tokens, excl. <s>
            # same logic inside neg_log_prob: start at 1 for bigram, 2 for trigram
            n_predicted = len(sent)-1 if n==2 else len(sent)-2
            total_tokens += n_predicted

        avg_neg_log_prob = total_neg_log_prob / total_tokens
        return math.exp(avg_neg_log_prob)

In [73]:
# Test on Tolkien
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(hobbit_encoded)

test_seq = hobbit_encoded[0]
print(test_seq)
print("bigram neg log prob:", ngram.neg_log_prob(test_seq, n=2, k=1.0))
print("trigram neg log prob:", ngram.neg_log_prob(test_seq, n=3, k=1.0))
print(bpe.decode(test_seq))

[2, 36, 49, 1532, 49, 40, 59, 51, 40, 38, 55, 40, 39, 578, 3999, 55, 60, 733, 262, 1982, 296, 263, 1103, 429, 1570, 262, 735, 17, 3]
bigram neg log prob: 193.13249070196224
trigram neg log prob: 207.5890976558037
AN UNEXPECTED PARTY In a hole in the ground there lived a hobbit.


In [74]:
# Test on Doyle
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(lostworld_encoded)

test_seq = lostworld_encoded[0]
print(test_seq)
print("bigram neg log prob:", ngram.neg_log_prob(test_seq, n=2, k=1.0))
print("trigram neg log prob:", ngram.neg_log_prob(test_seq, n=3, k=1.0))
print(bpe.decode(test_seq))

[2, 5, 1731, 4399, 398, 2832, 286, 946, 1224, 791, 419, 1532, 86, 5, 881, 17, 3]
bigram neg log prob: 114.26221001053686
trigram neg log prob: 114.25973565954511
"There Are Heroisms All Round Us" Mr.


*Note:* Abbreviation-splitting limitation. Finding "." + space + Capital Letter means that titles such as "Mr. Smith" are split into two sentences.

In [75]:
import random

for seq in random.sample(hobbit_encoded, 5):
    print(bpe.decode(seq))
    print()

for seq in random.sample(lostworld_encoded, 5):
    print(bpe.decode(seq))
    print()

Elvish singing is not a thing to miss, in June under the stars, not if you care for such things.

One right, yes.

He loved elves, though he seldom met them; but he was a little frightened of them too.

Then it was like a horrible game of blind-man’s-buff.

They were growing anxious, for they saw now that the house might be hidden almost anywhere between them and the mountains.

Would Mr.

I invite you to be present at the exhibition." He handed me a card from his desk. "You will perceive that Mr.

It was so high that I could not reach the top of it with my hand, and it appeared to be covered with grease.

As I emerged from the hall I was conscious for a moment of a rush of laughing students--down the pavement, and of an arm wielding a heavy umbrella, which rose and fell in the midst of them.

There's something in your line there, I am sure, and the Gazette should work it." "I really know nothing about him," said I. "I only remember his name in connection with the police-court proceedi

## Perplexity

In [76]:
# Load holdout sets
hobbit_holdout_path = os.path.join(base_dir, "texts", "hobbit_holdout.txt")
lostworld_holdout_path = os.path.join(base_dir, "texts", "lostworld_holdout.txt")

with open(hobbit_holdout_path, encoding='utf-8') as f:
    hobbit_holdout = f.read()

with open(lostworld_holdout_path, encoding='utf-8') as f:
    lostworld_holdout = f.read()

In [ ]:
# Test perplexity on holdout sets
# Tolkien
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(hobbit_encoded)
hobbit_holdout_sentences = get_sentences(hobbit_holdout)
hobbit_holdout_encoded = encode_sentences(hobbit_holdout_sentences, bpe)

bigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=2, k=1.0)
trigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=3, k=1.0)

print("Hobbit")
print("bigram perplexity:", bigram_ppl)
print("trigram perplexity:", trigram_ppl)

# Doyle
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(lostworld_encoded)
lostworld_holdout_sentences = get_sentences(lostworld_holdout)
lostworld_holdout_encoded = encode_sentences(lostworld_holdout_sentences, bpe)

bigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=2, k=1.0)
trigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=3, k=1.0)

print("\nLost World")
print("bigram perplexity:", bigram_ppl)
print("trigram perplexity:", trigram_ppl)

Hobbit
bigram perplexity: 915.2484905503446
trigram perplexity: 3244.3203950290495

Lost World
bigram perplexity: 1171.0410086794275
trigram perplexity: 3574.9187071801944


**Observations**
* Trigram performs much worse than bigram with k=1.0
* Number of distinct trigram contexts is huge compared to how many times each actually appears, so smoothing dominates probability estimate

In [84]:
# Testing k-values
k_vals = [1.0, 0.5, 0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001]

# Tolkien
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(hobbit_encoded)
print("Hobbit")
for k in k_vals:
    bigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=2, k=k)
    trigram_ppl = ngram.perplexity(hobbit_holdout_encoded, n=3, k=k)
    print(f"k={k:<6} bigram={bigram_ppl:9.2f}   trigram={trigram_ppl:9.2f}")

# Doyle
ngram = NGramModel(vocab_size=bpe.get_vocab_size())
ngram.train(lostworld_encoded)
print("\nLost World")
for k in k_vals:
    bigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=2, k=k)
    trigram_ppl = ngram.perplexity(lostworld_holdout_encoded, n=3, k=k)
    print(f"k={k:<6} bigram={bigram_ppl:9.2f}   trigram={trigram_ppl:9.2f}")

Hobbit
k=1.0    bigram=   915.25   trigram=  3244.32
k=0.5    bigram=   684.38   trigram=  2809.16
k=0.1    bigram=   387.96   trigram=  1947.10
k=0.01   bigram=   273.99   trigram=  1283.29
k=0.001  bigram=   346.57   trigram=  1237.18
k=0.0001 bigram=   628.70   trigram=  2010.28
k=1e-05  bigram=  1266.63   trigram=  4500.52
k=1e-06  bigram=  2591.69   trigram= 10824.14

Lost World
k=1.0    bigram=  1171.04   trigram=  3574.92
k=0.5    bigram=   879.95   trigram=  3168.45
k=0.1    bigram=   488.79   trigram=  2306.41
k=0.01   bigram=   326.04   trigram=  1582.17
k=0.001  bigram=   409.20   trigram=  1518.34
k=0.0001 bigram=   779.24   trigram=  2457.48
k=1e-05  bigram=  1676.60   trigram=  5623.86
k=1e-06  bigram=  3672.38   trigram= 13926.26


**Observations**
* The bigram models outperforms the trigram model at every value of k

*Bigrams*: perplexity drops from k=1.0 to k=0.01, then increases at k=0.001 for both books
* k = 0.01 is ideal hyperparameter for bigrams

*Trigrams*: perplexity drops from k=1.0 to k=0.001, then increases for both books
* k = 0.001 is ideal hyperparameter for trigrams

|Model|Best k|Best perplexity (Hobbit)|Best perplexity (Lost World)|
|---|---|---|---|
|Bigram|0.01|273.99|326.04|
|Trigram|0.001|1237.18|1518.34|